# L13 — Market Making: Simulación y Heurísticas

Implementaremos un market maker desde cero, mediremos el impacto del inventario en el P&L,
y compararemos estrategias naive vs skew. La clase `MarketMakingBacktest` que construyes aquí
la reutilizaremos en L14 para comparar con Avellaneda-Stoikov.

---

## Tiers de ejercicios

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Completa en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o para los más rápidos |

---

## Ejercicio 0 — Reflexión inicial (sin código)

1. Un market maker cotiza bid=99.95 y ask=100.05 (spread = 10 cents). En 1 hora recibe 200 fills, pero al final del día tiene un inventario neto de +50 lotes y el precio ha bajado 0.30. ¿El día fue rentable?

2. ¿Por qué un spread más ancho no siempre es mejor para el market maker?

3. Si siempre recibes más compras que ventas, ¿qué está pasando? ¿Es buena señal o mala?

4. ¿En qué se diferencia un market maker de un especulador direccional?

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

plt.rcParams.update({
    'figure.facecolor': '#09090b', 'axes.facecolor': '#18181b',
    'axes.edgecolor': '#27272a',   'axes.labelcolor': '#a1a1aa',
    'xtick.color': '#71717a',      'ytick.color': '#71717a',
    'text.color': '#f4f4f5',       'grid.color': '#27272a',
    'grid.linewidth': 0.5,
})

# Simulation parameters (fixed — do not change)
SIGMA    = 0.05    # price vol per step
SPREAD   = 0.10    # total spread in price units
KAPPA    = 5.0     # fill probability decay
ARR      = 1.0     # base arrival rate per step
SIGMA_SQ = SIGMA ** 2   # 0.0025
DT       = 1.0

print('Setup OK ✓')
print(f'SIGMA_SQ = {SIGMA_SQ}')
print(f'Fill prob at half-spread: exp(-kappa*half) = {np.exp(-KAPPA*SPREAD/2):.4f}')

---
## Ejercicio 1 — Precio browniano
*(Núcleo)*

Implementa `price_path(T, dt, sigma, seed=42)` que devuelve un array de precios siguiendo
un movimiento browniano aritmético con paso `sigma * sqrt(dt) * N(0,1)`, partiendo de 100.0.

In [ ]:
# Tu código aquí
def price_path(T, dt, sigma, seed=42):
    pass

In [ ]:
# ── Validación E1 ──────────────────────────────────────────────────
assert callable(price_path), 'price_path debe ser una función'
p = price_path(T=100, dt=1, sigma=SIGMA, seed=42)
assert len(p) == 101, f'Longitud esperada 101 (T+1), tienes {len(p)}'
assert abs(p[0] - 100.0) < 1e-9, 'El precio inicial debe ser 100.0'
assert abs(p[50]  - 100.22802758) < 0.001, f'p[50]  esperado ≈100.228, tienes {p[50]:.6f}'
assert abs(p[100] -  99.74865194) < 0.001, f'p[100] esperado ≈99.749, tienes {p[100]:.6f}'
print('✓ E1 correcto')
# Visualización rápida
long_path = price_path(T=3600, dt=1, sigma=SIGMA, seed=42)
fig, ax = plt.subplots(figsize=(10,3))
ax.plot(long_path, color='#22d3ee', linewidth=1)
ax.set_xlabel('Steps'); ax.set_ylabel('Precio'); ax.set_title('Precio browniano (T=3600)', color='#f4f4f5')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Solución E1 ───────────────────────────────────────────────────
def price_path(T, dt, sigma, seed=42):
    rng = np.random.default_rng(seed)
    n = int(T / dt)
    shocks = rng.normal(0, sigma * np.sqrt(dt), n)
    prices = np.zeros(n + 1)
    prices[0] = 100.0
    for i in range(n):
        prices[i+1] = prices[i] + shocks[i]
    return prices

---
## Ejercicio 2 — `NaiveMarketMaker`
*(Núcleo)*

Implementa la clase `NaiveMarketMaker` con los métodos:
- `__init__(spread, arr, seed)` — inicializa parámetros y estado
- `step(mid_price)` — avanza un step: computa bid/ask, genera fills por Poisson con prob `exp(-kappa*half)`
- `run(prices)` — itera sobre el array de precios llamando a `step()`
- `pnl` — propiedad: `cash + inventory * last_mid`

Usa la constante global `KAPPA`. Los fills llegan de forma independiente por cada lado.

In [ ]:
# Tu código aquí
class NaiveMarketMaker:
    def __init__(self, spread=SPREAD, arr=ARR, seed=42):
        pass
    
    def step(self, mid_price):
        pass
    
    def run(self, prices):
        pass
    
    @property
    def pnl(self):
        pass

In [ ]:
# ── Validación E2 ──────────────────────────────────────────────────
path_500 = price_path(T=500, dt=1, sigma=SIGMA, seed=42)
mm = NaiveMarketMaker(seed=42)
mm.run(path_500)
assert hasattr(mm, 'inventory'), 'Falta atributo inventory'
assert hasattr(mm, 'cash'), 'Falta atributo cash'
assert hasattr(mm, 'fills'), 'Falta atributo fills'
assert mm.fills == 774, f'Fills esperados 774, tienes {mm.fills}'
assert mm.inventory == -10, f'Inventario esperado -10, tienes {mm.inventory}'
assert abs(mm.pnl - 43.9333) < 0.1, f'PnL esperado ≈43.93, tienes {mm.pnl:.4f}'
print(f'✓ E2 correcto — fills={mm.fills}, inventory={mm.inventory}, pnl={mm.pnl:.4f}')

In [ ]:
# ── Solución E2 ───────────────────────────────────────────────────
class NaiveMarketMaker:
    def __init__(self, spread=SPREAD, arr=ARR, seed=42):
        self.spread = spread
        self.arr    = arr
        self.rng    = np.random.default_rng(seed)
        self.inventory = 0
        self.cash      = 0.0
        self.fills     = 0
        self._last_mid = 100.0
        self.inv_history = []
        self.pnl_history = []

    def step(self, mid_price):
        half = self.spread / 2
        bid  = mid_price - half
        ask  = mid_price + half
        p_fill = np.exp(-KAPPA * half) * self.arr * DT
        if self.rng.random() < p_fill:        # market buy hits ask → we sell
            self.inventory -= 1
            self.cash += ask
            self.fills += 1
        if self.rng.random() < p_fill:        # market sell hits bid → we buy
            self.inventory += 1
            self.cash -= bid
            self.fills += 1
        self._last_mid = mid_price
        self.inv_history.append(self.inventory)
        self.pnl_history.append(self.pnl)

    def run(self, prices):
        for p in prices[1:]:
            self.step(p)

    @property
    def pnl(self):
        return self.cash + self.inventory * self._last_mid

---
## Ejercicio 3 — `reservation_price(mid, q, gamma, sigma_sq)`
*(Núcleo)*

Implementa la función que calcula el precio de reserva:
$$r = \text{mid} - q \cdot \gamma \cdot \sigma^2$$

Esta es la intuición central de Avellaneda-Stoikov (sin el factor tiempo por ahora):
cuando estás largo (`q > 0`), el precio que "vale" para ti es menor que el mid de mercado.

In [ ]:
# Tu código aquí
def reservation_price(mid, q, gamma, sigma_sq):
    pass

In [ ]:
# ── Validación E3 ──────────────────────────────────────────────────
assert callable(reservation_price)
cases = [
    (100.0,  5,  0.1, SIGMA_SQ, 99.998750),
    (100.0, -3,  0.1, SIGMA_SQ, 100.000750),
    (100.0,  0,  0.1, SIGMA_SQ, 100.0),
    (100.0, 10,  0.2, SIGMA_SQ, 99.995000),
]
for mid, q, g, s2, expected in cases:
    result = reservation_price(mid, q, g, s2)
    assert abs(result - expected) < 1e-6, \
        f'reservation_price({mid}, {q}, {g}, {s2}): esperado {expected}, tienes {result}'
# Directionality checks
assert reservation_price(100, 5, 0.1, SIGMA_SQ) < 100, 'q>0 → r < mid'
assert reservation_price(100,-5, 0.1, SIGMA_SQ) > 100, 'q<0 → r > mid'
print('✓ E3 correcto — reservation_price OK')

In [ ]:
# ── Solución E3 ───────────────────────────────────────────────────
def reservation_price(mid, q, gamma, sigma_sq):
    return mid - q * gamma * sigma_sq

# Visualización: reservation price vs inventario
q_range = np.arange(-15, 16)
res_01 = [reservation_price(100.0, q, 0.1, SIGMA_SQ) for q in q_range]
res_03 = [reservation_price(100.0, q, 0.3, SIGMA_SQ) for q in q_range]

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(q_range, res_01, color='#22d3ee', linewidth=2, label='γ = 0.10')
ax.plot(q_range, res_03, color='#a78bfa', linewidth=2, label='γ = 0.30')
ax.axhline(100.0, color='#71717a', linestyle='--', linewidth=1, label='mid = 100')
ax.axvline(0, color='#27272a', linewidth=1)
ax.set_xlabel('Inventario q'); ax.set_ylabel('Reservation price')
ax.set_title('r = mid − q·γ·σ²', color='#f4f4f5')
ax.legend(framealpha=0.2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Ejercicio 4 — `SkewedMarketMaker`
*(Núcleo)*

Igual que `NaiveMarketMaker` pero usando `reservation_price` para desplazar las cotizaciones.

La lógica de fill cambia: la probabilidad de que te ejecuten en un lado depende de la
**distancia desde el mid** (no desde el reservation price), ya que el mercado cotiza
contra el mid.

```python
res  = reservation_price(mid, self.inventory, self.gamma, SIGMA_SQ)
bid  = res - half
ask  = res + half
# Fill prob: exp(-kappa * distancia_desde_mid)
p_bid = exp(-kappa * max(mid - bid, 0))
p_ask = exp(-kappa * max(ask - mid, 0))
```

In [ ]:
# Tu código aquí
class SkewedMarketMaker:
    def __init__(self, spread=SPREAD, gamma=0.1, arr=ARR, seed=42):
        pass
    
    def step(self, mid_price):
        pass
    
    def run(self, prices):
        pass
    
    @property
    def pnl(self):
        pass

In [ ]:
# ── Validación E4 ──────────────────────────────────────────────────
path_3600 = price_path(T=3600, dt=1, sigma=SIGMA, seed=42)
mm_s = SkewedMarketMaker(seed=42)
mm_s.run(path_3600)
assert hasattr(mm_s, 'inventory')
assert mm_s.fills == 5643, f'Fills esperados 5643, tienes {mm_s.fills}'
assert mm_s.inventory == -3, f'Inventario esperado -3, tienes {mm_s.inventory}'
assert abs(mm_s.pnl - 291.6679) < 1.0, f'PnL esperado ≈291.67, tienes {mm_s.pnl:.4f}'
print(f'✓ E4 correcto — skewed: fills={mm_s.fills}, inv={mm_s.inventory}, pnl={mm_s.pnl:.4f}')

In [ ]:
# ── Solución E4 ───────────────────────────────────────────────────
class SkewedMarketMaker:
    def __init__(self, spread=SPREAD, gamma=0.1, arr=ARR, seed=42):
        self.spread = spread
        self.gamma  = gamma
        self.arr    = arr
        self.rng    = np.random.default_rng(seed)
        self.inventory = 0
        self.cash      = 0.0
        self.fills     = 0
        self._last_mid = 100.0
        self.inv_history = []
        self.pnl_history = []

    def step(self, mid_price):
        half = self.spread / 2
        res  = reservation_price(mid_price, self.inventory, self.gamma, SIGMA_SQ)
        bid  = res - half
        ask  = res + half
        d_bid = max(mid_price - bid, 0)
        d_ask = max(ask - mid_price, 0)
        p_bid = np.exp(-KAPPA * d_bid) * self.arr * DT
        p_ask = np.exp(-KAPPA * d_ask) * self.arr * DT
        if self.rng.random() < p_ask:        # market buy hits ask → we sell
            self.inventory -= 1
            self.cash += ask
            self.fills += 1
        if self.rng.random() < p_bid:        # market sell hits bid → we buy
            self.inventory += 1
            self.cash -= bid
            self.fills += 1
        self._last_mid = mid_price
        self.inv_history.append(self.inventory)
        self.pnl_history.append(self.pnl)

    def run(self, prices):
        for p in prices[1:]:
            self.step(p)

    @property
    def pnl(self):
        return self.cash + self.inventory * self._last_mid

---
## Ejercicio 5 — Comparar Naive vs Skewed
*(Núcleo)*

Corre ambos market makers sobre el mismo `path_3600` (ya generado). Crea un gráfico
con dos subplots: inventario vs tiempo y P&L vs tiempo para ambos. Calcula y reporta:
- `std(inventario)` para cada uno
- `max_abs_inventory` para cada uno
- P&L final

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Validación E5 ──────────────────────────────────────────────────
# Run naive on same path for comparison
mm_n5 = NaiveMarketMaker(seed=42)
mm_n5.run(path_3600)
mm_s5 = SkewedMarketMaker(seed=42)
mm_s5.run(path_3600)
inv_n = mm_n5.inv_history
inv_s = mm_s5.inv_history
assert np.std(inv_s) < np.std(inv_n), \
    f'El skewed debe tener menor std de inventario. naive={np.std(inv_n):.2f} skewed={np.std(inv_s):.2f}'
assert max(abs(i) for i in inv_s) <= max(abs(i) for i in inv_n), \
    'El skewed debe tener menor max_abs_inventory'
print(f'✓ E5 correcto')
print(f'  Naive:  inv_std={np.std(inv_n):.2f}, max_abs={max(abs(i) for i in inv_n)}, pnl={mm_n5.pnl:.2f}')
print(f'  Skewed: inv_std={np.std(inv_s):.2f}, max_abs={max(abs(i) for i in inv_s)}, pnl={mm_s5.pnl:.2f}')

In [ ]:
# ── Solución E5 ───────────────────────────────────────────────────
mm_n5 = NaiveMarketMaker(seed=42)
mm_n5.run(path_3600)
mm_s5 = SkewedMarketMaker(seed=42)
mm_s5.run(path_3600)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = np.arange(len(mm_n5.inv_history))

ax1.plot(t, mm_n5.inv_history, color='#71717a', linewidth=1, label='Naive', alpha=0.8)
ax1.plot(t, mm_s5.inv_history, color='#22d3ee', linewidth=1.5, label='Skewed')
ax1.axhline(0, color='#27272a', linewidth=1)
ax1.set_ylabel('Inventario'); ax1.legend(framealpha=0.2); ax1.grid(True, alpha=0.3)
ax1.set_title('Naive vs Skewed Market Maker', color='#f4f4f5')

ax2.plot(t, mm_n5.pnl_history, color='#71717a', linewidth=1, label='Naive', alpha=0.8)
ax2.plot(t, mm_s5.pnl_history, color='#22d3ee', linewidth=1.5, label='Skewed')
ax2.set_xlabel('Step'); ax2.set_ylabel('P&L'); ax2.legend(framealpha=0.2); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f'Naive:  inv_std={np.std(mm_n5.inv_history):.2f}, max_abs={max(abs(i) for i in mm_n5.inv_history)}')
print(f'Skewed: inv_std={np.std(mm_s5.inv_history):.2f}, max_abs={max(abs(i) for i in mm_s5.inv_history)}')

---
## Ejercicio 6 — Shock de volatilidad
*(Si vamos bien)*

Genera un precio con un **shock de volatilidad** a mitad de simulación: entre los steps 1200–1500,
usa `sigma × 5`. Corre ambos market makers y visualiza. ¿Quién se recupera mejor?

Pista: modifica `price_path` para aceptar un parámetro `vol_shock=(1200, 1500, 5)`.

In [ ]:
# Tu código aquí
def price_path_with_shock(T, dt, sigma, shock_range=(1200, 1500), shock_mult=5.0, seed=42):
    pass

In [ ]:
# ── Validación E6 ──────────────────────────────────────────────────
assert callable(price_path_with_shock)
p_shock = price_path_with_shock(T=3600, seed=42)
# Shock should create more volatility in the specified range
normal_vol = np.std(np.diff(p_shock[:1200]))
shock_vol  = np.std(np.diff(p_shock[1200:1500]))
assert shock_vol > normal_vol * 2, \
    f'La volatilidad durante el shock ({shock_vol:.4f}) debe ser > 2× la normal ({normal_vol:.4f})'
print(f'✓ E6 correcto — vol normal={normal_vol:.4f}, vol shock={shock_vol:.4f} (ratio={shock_vol/normal_vol:.1f}×)')

In [ ]:
# ── Solución E6 ───────────────────────────────────────────────────
def price_path_with_shock(T, dt=1.0, sigma=SIGMA, shock_range=(1200, 1500), shock_mult=5.0, seed=42):
    rng = np.random.default_rng(seed)
    n = int(T / dt)
    prices = np.zeros(n + 1)
    prices[0] = 100.0
    for i in range(n):
        s = sigma * shock_mult if shock_range[0] <= i < shock_range[1] else sigma
        prices[i+1] = prices[i] + rng.normal(0, s * np.sqrt(dt))
    return prices

p_shock = price_path_with_shock(T=3600, seed=42)
mm_n6 = NaiveMarketMaker(seed=42);  mm_n6.run(p_shock)
mm_s6 = SkewedMarketMaker(seed=42); mm_s6.run(p_shock)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
t = np.arange(len(p_shock))
axes[0].plot(p_shock, color='#fbbf24', linewidth=1)
axes[0].axvspan(1200, 1500, color='rgba(248,113,113,0.1)', alpha=0.15, label='shock zone')
axes[0].set_ylabel('Precio'); axes[0].legend(framealpha=0.2); axes[0].grid(True, alpha=0.3)
axes[0].set_title('Precio con shock de volatilidad', color='#f4f4f5')

ti = np.arange(len(mm_n6.inv_history))
axes[1].plot(ti, mm_n6.inv_history, color='#71717a', linewidth=1, label='Naive', alpha=0.8)
axes[1].plot(ti, mm_s6.inv_history, color='#22d3ee', linewidth=1.5, label='Skewed')
axes[1].axvspan(1200, 1500, color='red', alpha=0.06)
axes[1].set_ylabel('Inventario'); axes[1].legend(framealpha=0.2); axes[1].grid(True, alpha=0.3)

axes[2].plot(ti, mm_n6.pnl_history, color='#71717a', linewidth=1, label='Naive', alpha=0.8)
axes[2].plot(ti, mm_s6.pnl_history, color='#22d3ee', linewidth=1.5, label='Skewed')
axes[2].axvspan(1200, 1500, color='red', alpha=0.06)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('P&L')
axes[2].legend(framealpha=0.2); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## Ejercicio 7 — Grid search de gamma
*(Si vamos bien)*

Para `SkewedMarketMaker`, encuentra el valor de `gamma` que maximiza el **Sharpe ratio** del P&L
sobre 20 simulaciones con diferentes semillas.

Sharpe ratio (simplificado) = `mean(pnl_final) / std(pnl_final)` sobre las 20 semillas.

Haz un gráfico de Sharpe vs gamma para `gamma ∈ [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]`.

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Solución E7 ───────────────────────────────────────────────────
gammas = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50]
N_SIM  = 20
T_SIM  = 3600

sharpes = []
for g in gammas:
    pnls = []
    for seed in range(N_SIM):
        path = price_path(T=T_SIM, dt=1, sigma=SIGMA, seed=seed)
        mm = SkewedMarketMaker(gamma=g, seed=seed)
        mm.run(path)
        pnls.append(mm.pnl)
    sharpes.append(np.mean(pnls) / (np.std(pnls) + 1e-9))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gammas, sharpes, color='#a78bfa', linewidth=2, marker='o', markersize=7)
ax.axhline(0, color='#27272a', linewidth=1)
best_g = gammas[np.argmax(sharpes)]
ax.axvline(best_g, color='#22d3ee', linestyle='--', alpha=0.6, label=f'γ* ≈ {best_g}')
ax.set_xlabel('γ (risk aversion)'); ax.set_ylabel('Sharpe ratio')
ax.set_title('Sharpe del SkewedMM vs γ', color='#f4f4f5')
ax.legend(framealpha=0.2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'γ óptimo empírico: {best_g}')

---
## Ejercicio 8 — P&L por componentes
*(Bonus / casa)*

Descompón el P&L del `NaiveMarketMaker` en:
- `gross_spread_pnl` = `fills × spread / 2` (ingresos brutos por spread)
- `inventory_pnl` = P&L atribuido al movimiento del precio sobre el inventario
- Verifica: `gross_spread_pnl + inventory_pnl ≈ pnl_final`

Modifica `NaiveMarketMaker` para trackear estas dos series y grafícalas.

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Solución E8 ───────────────────────────────────────────────────
class NaiveMarketMakerV2(NaiveMarketMaker):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gross_spread_pnl = 0.0
        self.gross_history    = []
        self.inv_pnl_history  = []
        self._entry_price_sum = 0.0  # track cost basis

    def step(self, mid_price):
        half = self.spread / 2
        bid  = mid_price - half
        ask  = mid_price + half
        p_fill = np.exp(-KAPPA * half) * self.arr * DT
        if self.rng.random() < p_fill:
            self.inventory += 1; self.cash -= bid; self.fills += 1
            self.gross_spread_pnl += half
        if self.rng.random() < p_fill:
            self.inventory -= 1; self.cash += ask; self.fills += 1
            self.gross_spread_pnl += half
        self._last_mid = mid_price
        inv_pnl = self.pnl - self.gross_spread_pnl
        self.inv_history.append(self.inventory)
        self.pnl_history.append(self.pnl)
        self.gross_history.append(self.gross_spread_pnl)
        self.inv_pnl_history.append(inv_pnl)

mmv2 = NaiveMarketMakerV2(seed=42)
mmv2.run(path_3600)

print(f'gross_spread_pnl = {mmv2.gross_spread_pnl:.2f}')
print(f'inventory_pnl    = {mmv2.inv_pnl_history[-1]:.2f}')
print(f'total pnl        = {mmv2.pnl:.2f}')
print(f'check: {mmv2.gross_spread_pnl + mmv2.inv_pnl_history[-1]:.2f} ≈ {mmv2.pnl:.2f}')

t = np.arange(len(mmv2.pnl_history))
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, mmv2.gross_history, color='#4ade80', linewidth=1.5, label='Gross spread PnL')
ax.plot(t, mmv2.inv_pnl_history, color='#f87171', linewidth=1, label='Inventory PnL', alpha=0.8)
ax.plot(t, mmv2.pnl_history, color='#22d3ee', linewidth=1.5, label='Total PnL')
ax.axhline(0, color='#27272a', linewidth=1)
ax.set_xlabel('Step'); ax.set_ylabel('P&L')
ax.set_title('Descomposición del P&L', color='#f4f4f5')
ax.legend(framealpha=0.2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Ejercicio 9 — Imbalance adjustment
*(Bonus / casa)*

Implementa `ImbalanceMarketMaker` que ajusta las cotizaciones según el imbalance del libro:

```python
adj  = (imbalance - 0.5) * alpha * mid
bid  = mid + adj - spread/2
ask  = mid + adj + spread/2
```

Simula el imbalance como un proceso con media-reversión hacia 0.5.
Compara con `NaiveMarketMaker` en Sharpe ratio sobre 20 semillas.

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Solución E9 ───────────────────────────────────────────────────
class ImbalanceMarketMaker:
    def __init__(self, spread=SPREAD, alpha=0.02, arr=ARR, seed=42):
        self.spread = spread; self.alpha = alpha; self.arr = arr
        self.rng    = np.random.default_rng(seed)
        self.inventory = 0; self.cash = 0.0; self.fills = 0
        self._last_mid = 100.0; self._imb = 0.5
        self.inv_history = []; self.pnl_history = []

    def step(self, mid):
        # Mean-reverting imbalance
        self._imb = np.clip(self._imb * 0.95 + 0.5 * 0.05 + (self.rng.random()-0.5)*0.08, 0.1, 0.9)
        adj  = (self._imb - 0.5) * self.alpha * mid
        half = self.spread / 2
        bid  = mid + adj - half
        ask  = mid + adj + half
        p_bid = np.exp(-KAPPA * max(mid - bid, 0)) * self.arr * DT
        p_ask = np.exp(-KAPPA * max(ask - mid, 0)) * self.arr * DT
        if self.rng.random() < p_bid: self.inventory += 1; self.cash -= bid; self.fills += 1
        if self.rng.random() < p_ask: self.inventory -= 1; self.cash += ask; self.fills += 1
        self._last_mid = mid
        self.inv_history.append(self.inventory)
        self.pnl_history.append(self.cash + self.inventory * mid)

    def run(self, prices):
        for p in prices[1:]: self.step(p)

    @property
    def pnl(self): return self.cash + self.inventory * self._last_mid

# Sharpe comparison
def sharpe(pnls): return np.mean(pnls) / (np.std(pnls) + 1e-9)

N = 20
pnls_naive, pnls_imb = [], []
for s in range(N):
    path = price_path(T=3600, dt=1, sigma=SIGMA, seed=s)
    mm_n = NaiveMarketMaker(seed=s); mm_n.run(path); pnls_naive.append(mm_n.pnl)
    mm_i = ImbalanceMarketMaker(seed=s); mm_i.run(path); pnls_imb.append(mm_i.pnl)

print(f'Naive sharpe:     {sharpe(pnls_naive):.3f}')
print(f'Imbalance sharpe: {sharpe(pnls_imb):.3f}')

---
## Ejercicio 10 — `MarketMakingBacktest` (clase reutilizable para L14)
*(Bonus / casa)*

Crea una clase `MarketMakingBacktest` que:
1. Acepta cualquier market maker (naive, skewed, imbalance…)
2. Corre N simulaciones con distintas semillas
3. Devuelve un DataFrame con `pnl_final`, `max_abs_inv`, `inv_std`, `fills`, `sharpe`
4. Tiene un método `plot_comparison([mm1, mm2], labels)` que compara visualmente dos estrategias

Esta clase la usaremos en L14 para comparar **Avellaneda-Stoikov vs Naive**.

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Validación E10 ─────────────────────────────────────────────────
assert 'MarketMakingBacktest' in dir() or 'MarketMakingBacktest' in globals()
bt = MarketMakingBacktest(T=1000, N=5)
results = bt.run(NaiveMarketMaker)
import pandas as pd
assert isinstance(results, pd.DataFrame), 'run() debe devolver un DataFrame'
assert 'pnl_final' in results.columns, 'Falta columna pnl_final'
assert 'inv_std' in results.columns, 'Falta columna inv_std'
assert len(results) == 5, f'Debe tener 5 filas (N=5), tienes {len(results)}'
print('✓ E10 correcto — MarketMakingBacktest funciona')
print(results.round(3))

In [ ]:
# ── Solución E10 ──────────────────────────────────────────────────
import pandas as pd

class MarketMakingBacktest:
    def __init__(self, T=3600, N=20):
        self.T = T
        self.N = N

    def run(self, MMClass, **kwargs):
        rows = []
        for seed in range(self.N):
            path = price_path(T=self.T, dt=1, sigma=SIGMA, seed=seed)
            mm = MMClass(seed=seed, **kwargs)
            mm.run(path)
            rows.append({
                'seed':        seed,
                'pnl_final':   round(mm.pnl, 4),
                'fills':       mm.fills,
                'inv_final':   mm.inventory,
                'max_abs_inv': max(abs(i) for i in mm.inv_history),
                'inv_std':     round(float(np.std(mm.inv_history)), 4),
            })
        df = pd.DataFrame(rows)
        df['sharpe'] = round(df['pnl_final'].mean() / (df['pnl_final'].std() + 1e-9), 4)
        return df

    def plot_comparison(self, mm_classes, labels, **kwargs):
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        colors = ['#71717a', '#22d3ee', '#a78bfa', '#fbbf24']
        for i, (MMCls, label) in enumerate(zip(mm_classes, labels)):
            df = self.run(MMCls, **kwargs)
            color = colors[i % len(colors)]
            axes[0].hist(df['pnl_final'], bins=10, color=color, alpha=0.6, label=label)
            axes[1].hist(df['inv_std'],   bins=10, color=color, alpha=0.6, label=label)
            axes[2].bar(i, df['sharpe'].iloc[0], color=color, alpha=0.8)
        for ax, title, xlabel in zip(axes, ['P&L final','Inventory std','Sharpe'], ['P&L','std','Strategy']):
            ax.set_title(title, color='#f4f4f5'); ax.set_xlabel(xlabel)
            ax.legend(framealpha=0.2) if xlabel != 'Strategy' else None
            ax.grid(True, alpha=0.3)
        axes[2].set_xticks(range(len(labels))); axes[2].set_xticklabels(labels)
        plt.tight_layout(); plt.show()

# Demo
bt = MarketMakingBacktest(T=3600, N=20)
bt.plot_comparison([NaiveMarketMaker, SkewedMarketMaker], ['Naive', 'Skewed'])